In [1]:
%pip install -U aif360 pandas numpy scikit-learn matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   --------------------------- ------------ 7.3/10.7 MB 39.1 MB/s eta 0:00:01
   ---------------------------------------- 10.7/10.7 MB 27.4 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ------------------------------ --------- 9.4/12.6 MB 52.1 MB/s eta 0:00:01
   ---------------------------------------  12.3/12.6 MB 49.6 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 30.8 MB/s  0:00:00
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------  8.1/8.2 MB 45.5 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 28.5 MB/s  0:00:00
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ---------------------------------------  9.2/9.3 MB 48.9 MB/s eta 0:00:01
   --------------------------


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# Assignment 4 - Task 2
# Evaluation of the Reweighing bias-mitigation algorithm
#
# Datasets:
# 1. Adult Census Income
# 2. German Credit
# 3. COMPAS
#
# Algorithm:
# Reweighing (pre-processing)
#
# Classifier:
# Logistic Regression
# ============================================================


# ------------------------------------------------------------
# 1. IMPORT PACKAGES
# ------------------------------------------------------------

import sys
import platform
import warnings
import numpy as np
import pandas as pd
import sklearn
import aif360

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.neighbors import NearestNeighbors

from aif360.sklearn.datasets import (
    fetch_adult,
    fetch_german,
    fetch_compas,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_SIZE = 0.30
N_NEIGHBORS = 5


# ------------------------------------------------------------
# 2. HELPER FUNCTION: FIND PROTECTED ATTRIBUTE
# ------------------------------------------------------------

def extract_protected_attribute(X, attribute_name):
    """
    Extract a protected attribute from either:
    - the index of the AIF360 dataset; or
    - a regular dataframe column.
    """

    if attribute_name in X.index.names:
        protected = pd.Series(
            X.index.get_level_values(attribute_name),
            index=X.index,
            name=attribute_name
        )

    elif attribute_name in X.columns:
        protected = X[attribute_name].copy()

    else:
        raise ValueError(
            f"Protected attribute '{attribute_name}' was not found. "
            f"Index names: {X.index.names}; "
            f"Columns: {list(X.columns)}"
        )

    return protected.astype(str)


# ------------------------------------------------------------
# 3. HELPER FUNCTION: REMOVE PROTECTED FEATURE
# ------------------------------------------------------------

def remove_protected_feature(X, attribute_name):
    """
    Remove the protected attribute from the predictor columns
    if it is present there.

    The attribute remains separately available for:
    - fairness measurement;
    - Reweighing calculations.
    """

    X_model = X.copy()

    if attribute_name in X_model.columns:
        X_model = X_model.drop(columns=[attribute_name])

    return X_model


# ------------------------------------------------------------
# 4. HELPER FUNCTION: CREATE BINARY LABEL
# ------------------------------------------------------------

def create_binary_label(y, favorable_label):
    """
    Convert the favorable outcome to 1 and all other outcomes to 0.
    """

    y_string = pd.Series(
        np.asarray(y).reshape(-1),
        index=y.index if hasattr(y, "index") else None
    ).astype(str)

    y_binary = (
        y_string.str.strip().str.lower()
        == favorable_label.strip().lower()
    ).astype(int)

    return y_binary


# ------------------------------------------------------------
# 5. FAIRNESS METRICS
# ------------------------------------------------------------

def group_fairness_metrics(y, protected, privileged_value):
    """
    Calculate:
    - positive outcome rate for privileged group;
    - positive outcome rate for unprivileged group;
    - demographic parity difference;
    - disparate impact ratio.

    DP_Diff = unprivileged rate - privileged rate
    Ideal value = 0

    DIR = unprivileged rate / privileged rate
    Ideal value = 1
    """

    y = np.asarray(y).astype(int)
    protected = np.asarray(protected).astype(str)

    privileged_mask = (
        np.char.lower(protected)
        == privileged_value.lower()
    )

    unprivileged_mask = ~privileged_mask

    if privileged_mask.sum() == 0:
        raise ValueError(
            f"No privileged observations found for value "
            f"'{privileged_value}'. Values found: "
            f"{np.unique(protected)}"
        )

    if unprivileged_mask.sum() == 0:
        raise ValueError(
            "No unprivileged observations found."
        )

    privileged_rate = y[privileged_mask].mean()
    unprivileged_rate = y[unprivileged_mask].mean()

    dp_difference = (
        unprivileged_rate - privileged_rate
    )

    if privileged_rate == 0:
        disparate_impact_ratio = np.nan
    else:
        disparate_impact_ratio = (
            unprivileged_rate / privileged_rate
        )

    return {
        "Privileged_Positive_Rate": privileged_rate,
        "Unprivileged_Positive_Rate": unprivileged_rate,
        "DP_Diff": dp_difference,
        "DIR": disparate_impact_ratio,
    }


# ------------------------------------------------------------
# 6. CONSISTENCY
# ------------------------------------------------------------

def consistency_score_fast(X, y, n_neighbors=5):
    """
    Calculate the individual-fairness consistency score.

    Higher values are better.
    Ideal value = 1.

    For each observation, the label is compared with the mean
    label of its nearest neighbours.
    """

    y = np.asarray(y).astype(float)

    neighbour_model = NearestNeighbors(
        n_neighbors=n_neighbors + 1,
        metric="euclidean",
        n_jobs=-1
    )

    neighbour_model.fit(X)

    neighbour_indices = neighbour_model.kneighbors(
        X,
        return_distance=False
    )

    # First neighbour is normally the observation itself.
    neighbour_indices = neighbour_indices[:, 1:]

    neighbour_means = y[neighbour_indices].mean(axis=1)

    consistency = 1 - np.mean(
        np.abs(y - neighbour_means)
    )

    return float(consistency)


# ------------------------------------------------------------
# 7. REWEIGHING IMPLEMENTATION
# ------------------------------------------------------------

def calculate_reweighing_weights(
    y_train,
    protected_train,
    privileged_value
):
    """
    Implement the Reweighing formula:

    w(a, y) = P(A=a) * P(Y=y) / P(A=a, Y=y)

    The calculation is based only on the training data.
    """

    y_train = np.asarray(y_train).astype(int)
    protected_train = np.asarray(
        protected_train
    ).astype(str)

    group = np.where(
        np.char.lower(protected_train)
        == privileged_value.lower(),
        1,  # privileged
        0   # unprivileged
    )

    n = len(y_train)
    weights = np.ones(n, dtype=float)
    weight_table = []

    for group_value in [0, 1]:
        for label_value in [0, 1]:

            group_mask = group == group_value
            label_mask = y_train == label_value
            joint_mask = group_mask & label_mask

            group_count = group_mask.sum()
            label_count = label_mask.sum()
            joint_count = joint_mask.sum()

            if joint_count == 0:
                raise ValueError(
                    "A protected-group/label combination has "
                    "zero observations. Reweighing cannot "
                    "calculate a valid weight."
                )

            probability_group = group_count / n
            probability_label = label_count / n
            probability_joint = joint_count / n

            weight = (
                probability_group
                * probability_label
                / probability_joint
            )

            weights[joint_mask] = weight

            weight_table.append({
                "Group": (
                    "Privileged"
                    if group_value == 1
                    else "Unprivileged"
                ),
                "Label": label_value,
                "Observed_Count": joint_count,
                "Weight": weight
            })

    return weights, pd.DataFrame(weight_table)


# ------------------------------------------------------------
# 8. PREPROCESSING
# ------------------------------------------------------------

def build_preprocessor(X):
    """
    Standardize numerical variables and one-hot encode
    categorical variables.
    """

    numeric_columns = X.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    categorical_columns = [
        column
        for column in X.columns
        if column not in numeric_columns
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                numeric_columns
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ),
                categorical_columns
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0
    )

    return preprocessor


# ------------------------------------------------------------
# 9. LOAD THE THREE DATASETS
# ------------------------------------------------------------

adult = fetch_adult(
    subset="all",
    binary_race=True,
    dropna=True
)

german = fetch_german(
    binary_age=True,
    dropna=True
)

compas = fetch_compas(
    subset="all",
    binary_race=True,
    dropna=True
)


# Dataset configuration
#
# Protected attribute choices:
# Adult: sex
# German Credit: sex
# COMPAS: race
#
# Favorable outcomes:
# Adult: income >50K
# German: good credit risk
# COMPAS: no recidivism ("Survived")

DATASETS = {
    "Adult": {
        "data": adult,
        "protected_attribute": "sex",
        "privileged_value": "Male",
        "favorable_label": ">50K",
    },

    "German Credit": {
        "data": german,
        "protected_attribute": "sex",
        "privileged_value": "male",
        "favorable_label": "good",
    },

    "COMPAS": {
        "data": compas,
        "protected_attribute": "race",
        "privileged_value": "Caucasian",
        "favorable_label": "Survived",
    },
}


# ------------------------------------------------------------
# 10. RUN THE COMPLETE PIPELINE
# ------------------------------------------------------------

all_results = []
all_weight_tables = []

for dataset_name, config in DATASETS.items():

    print("\n" + "=" * 70)
    print(f"DATASET: {dataset_name}")
    print("=" * 70)

    dataset = config["data"]

    X_original = dataset.X.copy()
    y_original = dataset.y.copy()

    protected_attribute = config["protected_attribute"]
    privileged_value = config["privileged_value"]
    favorable_label = config["favorable_label"]

    # Extract group membership
    protected = extract_protected_attribute(
        X_original,
        protected_attribute
    )

    # Create binary favorable/unfavorable label
    y_binary = create_binary_label(
        y_original,
        favorable_label
    )

    # Remove protected attribute as a direct predictor
    X_model = remove_protected_feature(
        X_original,
        protected_attribute
    )

    # Reset indices while preserving the same row order
    X_model = X_model.reset_index(drop=True)
    y_binary = y_binary.reset_index(drop=True)
    protected = protected.reset_index(drop=True)

    print(f"Rows: {len(X_model)}")
    print(f"Predictors before encoding: {X_model.shape[1]}")
    print(
        "Protected groups:",
        protected.value_counts().to_dict()
    )
    print(
        "Binary labels:",
        y_binary.value_counts().to_dict()
    )


    # --------------------------------------------------------
    # A. BEFORE MITIGATION:
    # fairness of original dataset labels
    # --------------------------------------------------------

    before_group_metrics = group_fairness_metrics(
        y=y_binary,
        protected=protected,
        privileged_value=privileged_value
    )

    # Encode the complete original dataset only for
    # the original-label consistency calculation.
    before_preprocessor = build_preprocessor(X_model)
    X_before_encoded = before_preprocessor.fit_transform(
        X_model
    )

    before_consistency = consistency_score_fast(
        X_before_encoded,
        y_binary,
        n_neighbors=N_NEIGHBORS
    )

    print("\nOriginal-label fairness:")
    print(
        f"DP_Diff: {before_group_metrics['DP_Diff']:.4f}"
    )
    print(
        f"DIR: {before_group_metrics['DIR']:.4f}"
    )
    print(
        f"Consistency: {before_consistency:.4f}"
    )


    # --------------------------------------------------------
    # B. STRATIFIED TRAIN/TEST SPLIT
    #
    # Stratify jointly by label and protected group so that
    # group-label combinations remain represented.
    # --------------------------------------------------------

    joint_stratification = (
        protected.astype(str)
        + "_"
        + y_binary.astype(str)
    )

    (
        X_train,
        X_test,
        y_train,
        y_test,
        protected_train,
        protected_test
    ) = train_test_split(
        X_model,
        y_binary,
        protected,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=joint_stratification
    )


    # --------------------------------------------------------
    # C. REWEIGH TRAINING DATA ONLY
    # --------------------------------------------------------

    train_weights, weight_table = (
        calculate_reweighing_weights(
            y_train=y_train,
            protected_train=protected_train,
            privileged_value=privileged_value
        )
    )

    weight_table.insert(
        0,
        "Dataset",
        dataset_name
    )

    all_weight_tables.append(weight_table)

    print("\nReweighing weights:")
    print(weight_table.to_string(index=False))


    # --------------------------------------------------------
    # D. FIT FEATURE PREPROCESSING ON TRAINING DATA ONLY
    # --------------------------------------------------------

    model_preprocessor = build_preprocessor(X_train)

    X_train_encoded = model_preprocessor.fit_transform(
        X_train
    )

    X_test_encoded = model_preprocessor.transform(
        X_test
    )


    # --------------------------------------------------------
    # E. TRAIN CLASSIFIER WITH REWEIGHING WEIGHTS
    # --------------------------------------------------------

    classifier = LogisticRegression(
        solver="liblinear",
        max_iter=1000,
        random_state=RANDOM_STATE
    )

    classifier.fit(
        X_train_encoded,
        y_train,
        sample_weight=train_weights
    )


    # --------------------------------------------------------
    # F. PREDICT ON UNCHANGED HELD-OUT TEST DATA
    # --------------------------------------------------------

    y_prediction = classifier.predict(
        X_test_encoded
    )


    # --------------------------------------------------------
    # G. AFTER MITIGATION:
    # fairness of final test predictions
    # --------------------------------------------------------

    after_group_metrics = group_fairness_metrics(
        y=y_prediction,
        protected=protected_test,
        privileged_value=privileged_value
    )

    after_consistency = consistency_score_fast(
        X_test_encoded,
        y_prediction,
        n_neighbors=N_NEIGHBORS
    )


    # --------------------------------------------------------
    # H. PREDICTIVE PERFORMANCE
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_prediction
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        y_prediction
    )

    precision = precision_score(
        y_test,
        y_prediction,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_prediction,
        zero_division=0
    )


    # --------------------------------------------------------
    # I. STORE RESULTS
    # --------------------------------------------------------

    all_results.append({
        "Dataset": dataset_name,

        "Protected_Attribute":
            protected_attribute,

        "Privileged_Group":
            privileged_value,

        "Favorable_Label":
            favorable_label,

        "Observations":
            len(X_model),

        "Train_Size":
            len(X_train),

        "Test_Size":
            len(X_test),

        "Before_DP_Diff":
            before_group_metrics["DP_Diff"],

        "After_DP_Diff":
            after_group_metrics["DP_Diff"],

        "DP_Diff_Improvement":
            abs(before_group_metrics["DP_Diff"])
            - abs(after_group_metrics["DP_Diff"]),

        "Before_DIR":
            before_group_metrics["DIR"],

        "After_DIR":
            after_group_metrics["DIR"],

        "DIR_Distance_Improvement":
            abs(before_group_metrics["DIR"] - 1)
            - abs(after_group_metrics["DIR"] - 1),

        "Before_Consistency":
            before_consistency,

        "After_Consistency":
            after_consistency,

        "Accuracy":
            accuracy,

        "Balanced_Accuracy":
            balanced_accuracy,

        "Precision":
            precision,

        "Recall":
            recall,

        "F1":
            f1,
    })

    print("\nFinal held-out test results:")
    print(
        f"DP_Diff: "
        f"{after_group_metrics['DP_Diff']:.4f}"
    )
    print(
        f"DIR: "
        f"{after_group_metrics['DIR']:.4f}"
    )
    print(
        f"Consistency: "
        f"{after_consistency:.4f}"
    )
    print(
        f"Accuracy: {accuracy:.4f}"
    )
    print(
        f"Balanced accuracy: "
        f"{balanced_accuracy:.4f}"
    )
    print(
        f"F1: {f1:.4f}"
    )


# ------------------------------------------------------------
# 11. FINAL RESULTS TABLE
# ------------------------------------------------------------

results = pd.DataFrame(all_results)

display_columns = [
    "Dataset",
    "Before_DP_Diff",
    "After_DP_Diff",
    "Before_DIR",
    "After_DIR",
    "Before_Consistency",
    "After_Consistency",
    "Accuracy",
    "Balanced_Accuracy",
    "F1",
]

final_table = results[display_columns].copy()

numeric_columns = final_table.select_dtypes(
    include="number"
).columns

final_table[numeric_columns] = (
    final_table[numeric_columns].round(4)
)

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)

display(final_table)


# ------------------------------------------------------------
# 12. SAVE RESULTS
# ------------------------------------------------------------

results.to_csv(
    "reweighing_full_results.csv",
    index=False
)

final_table.to_csv(
    "reweighing_report_table.csv",
    index=False
)

weight_results = pd.concat(
    all_weight_tables,
    ignore_index=True
)

weight_results.to_csv(
    "reweighing_weights.csv",
    index=False
)


# ------------------------------------------------------------
# 13. SAVE PARAMETERS AND SOFTWARE VERSIONS
# ------------------------------------------------------------

parameters = pd.DataFrame({
    "Parameter": [
        "Algorithm",
        "Algorithm category",
        "Classifier",
        "Test size",
        "Random state",
        "Stratification",
        "Consistency neighbours",
        "Logistic Regression solver",
        "Logistic Regression max_iter",
        "Protected feature in classifier",
    ],

    "Value": [
        "Reweighing",
        "Pre-processing",
        "Logistic Regression",
        TEST_SIZE,
        RANDOM_STATE,
        "Joint label + protected group",
        N_NEIGHBORS,
        "liblinear",
        1000,
        "Removed",
    ]
})

parameters.to_csv(
    "reweighing_parameters.csv",
    index=False
)

software_versions = pd.DataFrame({
    "Software": [
        "Python",
        "Platform",
        "AIF360",
        "pandas",
        "NumPy",
        "scikit-learn",
    ],

    "Version": [
        sys.version.replace("\n", " "),
        platform.platform(),
        aif360.__version__,
        pd.__version__,
        np.__version__,
        sklearn.__version__,
    ]
})

software_versions.to_csv(
    "software_versions.csv",
    index=False
)

print("\nSaved files:")
print("- reweighing_full_results.csv")
print("- reweighing_report_table.csv")
print("- reweighing_weights.csv")
print("- reweighing_parameters.csv")
print("- software_versions.csv")


DATASET: Adult
Rows: 45222
Predictors before encoding: 12
Protected groups: {'Male': 30527, 'Female': 14695}
Binary labels: {0: 34014, 1: 11208}

Original-label fairness:
DP_Diff: -0.1989
DIR: 0.3635
Consistency: 0.8035

Reweighing weights:
Dataset        Group  Label  Observed_Count   Weight
  Adult Unprivileged      0            9118 0.848524
  Adult Unprivileged      1            1168 2.182500
  Adult   Privileged      0           14692 1.094008
  Adult   Privileged      1            6677 0.793147

Final held-out test results:
DP_Diff: -0.1001
DIR: 0.5443
Consistency: 0.9482
Accuracy: 0.8444
Balanced accuracy: 0.7502
F1: 0.6423

DATASET: German Credit
Rows: 1000
Predictors before encoding: 20
Protected groups: {'male': 690, 'female': 310}
Binary labels: {1: 700, 0: 300}

Original-label fairness:
DP_Diff: -0.0748
DIR: 0.8966
Consistency: 0.6788

Reweighing weights:
      Dataset        Group  Label  Observed_Count   Weight
German Credit Unprivileged      0              76 0.856579
G

,Dataset,Before_DP_Diff,After_DP_Diff,Before_DIR,After_DIR,Before_Consistency,After_Consistency,Accuracy,Balanced_Accuracy,F1
0,Adult,-0.1989,-0.1001,0.3635,0.5443,0.8035,0.9482,0.8444,0.7502,0.6423
1,German Credit,-0.0748,-0.0117,0.8966,0.9845,0.6788,0.7733,0.7633,0.6944,0.8368
2,COMPAS,-0.1321,-0.2354,0.7830,0.6762,0.5992,0.8680,0.6618,0.6574,0.6965



Saved files:
- reweighing_full_results.csv
- reweighing_report_table.csv
- reweighing_weights.csv
- reweighing_parameters.csv
- software_versions.csv
